# Gmail Tool Flow Demo

This notebook exercises the Gmail-related source and tool flow against real Gmail reads.

It covers:
- `GmailEmailSource`
- `GmailFetchTool`
- `MailMindEmailClassifier` via `EmailClassifierTool`
- `EmailSummaryTool`
- `DraftReplyTool`
- `NotificationTool`

The Gmail connection uses `.env` values:
- `MAILMIND_GMAIL_CLIENT_ID`
- `MAILMIND_GMAIL_CLIENT_SECRET`

The OAuth token is written to `data/gmail_token.json`.

The classifier uses the real local LLM path from `src.llm.factory.LLMFactory`.
By default it uses `Qwen/Qwen3-1.7B`, or `EASY_AGENT_LLM_MODEL_NAME` if set.
On macOS, this notebook defaults the classifier device to `cpu` to avoid PyTorch MPS attention crashes; override with `EASY_AGENT_LLM_DEVICE_MAP` if needed.

Note: this notebook will stop early if the Gmail client secret is missing or empty.


In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


def load_dotenv(dotenv_path: Path) -> None:
    if not dotenv_path.exists():
        return
    for raw_line in dotenv_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        normalized_key = key.strip()
        normalized_value = value.strip().strip("'").strip('"')
        if not os.environ.get(normalized_key):
            os.environ[normalized_key] = normalized_value


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

load_dotenv(repo_root / ".env")
print(f"Using repo root: {repo_root}")


Using repo root: /Users/saketm10/Projects/easy_agents


In [2]:
required = {
    "MAILMIND_GMAIL_CLIENT_ID": os.getenv("MAILMIND_GMAIL_CLIENT_ID"),
    "MAILMIND_GMAIL_CLIENT_SECRET": os.getenv("MAILMIND_GMAIL_CLIENT_SECRET"),
}
missing = [key for key, value in required.items() if not value]
if missing:
    raise RuntimeError(
        f"Missing required .env keys: {missing}. Populate them before running the real Gmail demo."
    )

print("Found Gmail OAuth client credentials in .env")


Found Gmail OAuth client credentials in .env


In [3]:
from datetime import timezone

from agents.mailmind.helpers import MailMindEmailClassifier
from src.llm.factory import LLMFactory
from src.interfaces.email import Notifier
from src.schemas.domain import (
    ApprovalItem,
    ApprovalKind,
    ApprovalStatus,
    NotificationAttempt,
    NotificationStatus,
)
from src.sources.gmail import GmailEmailSource
from src.storage.duckdb_store import DuckDBMessageRepository
from src.tools.gmail import (
    DraftReplyTool,
    EmailClassifierTool,
    EmailSummaryTool,
    GmailFetchTool,
    NotificationTool,
)

class NotebookNotifier(Notifier):
    def send(self, payload):
        return NotificationAttempt(
            message_id=payload.message_id,
            channel=payload.channel,
            destination=payload.destination,
            payload=payload.model_dump(mode="json"),
            status=NotificationStatus.SENT,
        )


In [4]:
token_path = repo_root / "data" / "gmail_token.json"
source = GmailEmailSource(
    client_id=os.environ["MAILMIND_GMAIL_CLIENT_ID"],
    client_secret=os.environ["MAILMIND_GMAIL_CLIENT_SECRET"],
    token_path=token_path,
    max_results=5,
)

messages = source.fetch_new_messages()
print(f"Fetched {len(messages)} Gmail messages")
for message in messages:
    print({
        "source_id": message.source_id,
        "from_email": message.from_email,
        "subject": message.subject,
        "received_at": message.received_at.astimezone(timezone.utc).isoformat(),
    })


Fetched 5 Gmail messages
{'source_id': '19d7027e183c9620', 'from_email': 'information@mailers.hdfcbank.bank.in', 'subject': "HDFC Bank Update - Convert 💳 xx6054 spends of Rs. 50242 into EMI's!", 'received_at': '2026-04-09T02:50:58+00:00'}
{'source_id': '19d70247741290f2', 'from_email': 'email.campaign@sg.booking.com', 'subject': 'Save at least 15% on your summer stay ☀️', 'received_at': '2026-04-09T02:48:48+00:00'}
{'source_id': '19d70193ca329b1e', 'from_email': 'information@mailers.hdfcbank.bank.in', 'subject': 'Unlock Your LIFETIME FREE Credit Card & Get EXTRA Rs.250 Voucher!', 'received_at': '2026-04-09T02:33:47+00:00'}
{'source_id': '19d7010f5978bf14', 'from_email': 'sales@updates.clickastro.com', 'subject': 'Soumya Mohanty! What if your marriage destiny isn’t what you think? 🤔', 'received_at': '2026-04-09T02:27:32+00:00'}
{'source_id': '19d70049366033ff', 'from_email': 'alerts@hdfcbank.bank.in', 'subject': '❗  You have done a UPI txn. Check details!', 'received_at': '2026-04-09T02

In [5]:
db_path = repo_root / "data" / f"gmail_tool_demo_{os.getpid()}.duckdb"
repository = DuckDBMessageRepository(db_path)
repository.init_db()


fetch_tool = GmailFetchTool(source=source, repository=repository)
fetch_result = fetch_tool.execute(fetch_tool.input_schema(process_messages=True))
print(fetch_result.model_dump(mode="json"))


{'fetched_count': 5, 'processed_count': 5, 'emails': [{'id': '70db76ac-5fde-4d24-b90b-084f14707d7f', 'source_id': '19d7027e183c9620', 'from_email': 'information@mailers.hdfcbank.bank.in', 'from_name': 'HDFC Bank', 'subject': "HDFC Bank Update - Convert 💳 xx6054 spends of Rs. 50242 into EMI's!", 'received_at': '2026-04-09T02:50:58Z', 'category': None, 'priority_score': None, 'summary': None}, {'id': '86f64b84-08de-4dbf-af8a-cab85929cc60', 'source_id': '19d70247741290f2', 'from_email': 'email.campaign@sg.booking.com', 'from_name': 'Booking.com', 'subject': 'Save at least 15% on your summer stay ☀️', 'received_at': '2026-04-09T02:48:48Z', 'category': None, 'priority_score': None, 'summary': None}, {'id': 'c264c3ea-06ad-4a8d-8746-033b24d7105d', 'source_id': '19d70193ca329b1e', 'from_email': 'information@mailers.hdfcbank.bank.in', 'from_name': 'HDFC Bank', 'subject': 'Unlock Your LIFETIME FREE Credit Card & Get EXTRA Rs.250 Voucher!', 'received_at': '2026-04-09T02:33:47Z', 'category': None,

In [6]:
import time

classifier_model_name = os.getenv("EASY_AGENT_LLM_MODEL_NAME", "Qwen/Qwen3-1.7B")
classifier_device_map = os.getenv("EASY_AGENT_LLM_DEVICE_MAP", "cpu")
debug_prompt_path = "/tmp/mailmind_classifier_chat_prompt.txt"
debug_system_prompt_path = "/tmp/mailmind_classifier_system_prompt.txt"
debug_user_prompt_path = "/tmp/mailmind_classifier_user_prompt.txt"

t0 = time.perf_counter()
classifier = MailMindEmailClassifier(
    llm=LLMFactory.build_default_local_llm(
        model_name=classifier_model_name,
        device_map=classifier_device_map,
        max_new_tokens=1024,
        use_kv_chache=True
    )
)
classifier.llm.client.debug_prompt_path = debug_prompt_path
print(f"Using classifier model: {classifier_model_name} on {classifier_device_map}")
print(f"classifier build took {time.perf_counter() - t0:.2f}s")

message_id = repository.list_messages()[0].message.id
message = repository.get_message(message_id)
payload = classifier._build_payload(message)

t1 = time.perf_counter()
rendered_system_prompt = classifier.system_prompt.format(
    classification_classes_json=classifier._to_json(classifier.classification_classes),
    class_details_json=classifier._to_json(classifier.class_details),
    classification_mode="multi_label" if classifier.multi_label else "single_label",
    calc_prob=classifier.calc_prob,
)
rendered_user_prompt = classifier.user_prompt.format(
    input_json=classifier._to_json(payload),
    classification_classes_json=classifier._to_json(classifier.classification_classes),
    class_details_json=classifier._to_json(classifier.class_details),
    classification_mode="multi_label" if classifier.multi_label else "single_label",
    calc_prob=classifier.calc_prob,
)
rendered_system_prompt = classifier._append_mode_instruction(rendered_system_prompt)
rendered_user_prompt = classifier._append_mode_instruction(rendered_user_prompt)
structured_user_prompt = classifier.llm._build_structured_user_prompt(
    rendered_user_prompt,
    classifier.output_schema,
)
print(f"prompt rendering took {time.perf_counter() - t1:.2f}s")

with open(debug_system_prompt_path, "w", encoding="utf-8") as f:
    f.write(rendered_system_prompt)
with open(debug_user_prompt_path, "w", encoding="utf-8") as f:
    f.write(structured_user_prompt)

print("message_id:", message_id)
print("saved system prompt:", debug_system_prompt_path)
print("saved user prompt:", debug_user_prompt_path)
print("chat prompt will be saved to:", debug_prompt_path)
print("system prompt chars:", len(rendered_system_prompt))
print("user prompt chars:", len(structured_user_prompt))


Using classifier model: Qwen/Qwen3-1.7B on cpu
classifier build took 0.00s
prompt rendering took 0.00s
message_id: 70db76ac-5fde-4d24-b90b-084f14707d7f
saved system prompt: /tmp/mailmind_classifier_system_prompt.txt
saved user prompt: /tmp/mailmind_classifier_user_prompt.txt
chat prompt will be saved to: /tmp/mailmind_classifier_chat_prompt.txt
system prompt chars: 5182
user prompt chars: 26098


In [7]:
import time

t0 = time.perf_counter()
manual_result = classifier.llm.client.generate_result(
    rendered_system_prompt,
    structured_user_prompt,
)
elapsed = time.perf_counter() - t0

print(f"manual single-message generate_result took {elapsed:.2f}s")
print("chat prompt saved to:", debug_prompt_path)
print("\nRAW OUTPUT:\n")
print(manual_result.raw_text)
print("\nCONTENT:\n")
print(manual_result.content)
print("\nTHINKING:\n")
print(manual_result.thinking_content)


iteration : 658 | next_token_id: 92 | decoded: '}'
iteration : 659 | next_token_id: 151645 | decoded: '<|im_end|>'
manual single-message generate_result took 447.19s
chat prompt saved to: /tmp/mailmind_classifier_chat_prompt.txt

RAW OUTPUT:

<think>
Okay, let's tackle this email classification. First, I need to look at the email content provided. The subject line is "HDFC Bank Update - Convert xx6054 spends of Rs. 50242 into EMI's!" and the body mentions a promotional offer to convert credit card spends into EMI with a fee of Rs. 4699 per month for 12 months. There's a link provided for viewing the message.

Now, checking the allowed categories: strong_ml_research_job, deep_tech_opportunity, network_event, time_sensitive_professional, alumni_reconnect, personal_exceptional, promotion, newsletter, weak_recruiter, other.

The email is a promotion because it's a marketing message offering a service. The key points are the promotional offer, the fee, and the link. The subject mentions con

In [ ]:
import time

classifier_tool = EmailClassifierTool(repository=repository, classifier=classifier)
message_ids = classifier_tool.input_schema().message_ids or [
    bundle.message.id
    for bundle in repository.list_messages()
    if bundle.classification is None
]

print("message count:", len(message_ids))
tool_started = time.perf_counter()
timings = []
results = []
saved_message_ids = []

for index, message_id in enumerate(message_ids, start=1):
    message = repository.get_message(message_id)
    if message is None:
        print(f"[{index}/{len(message_ids)}] missing message {message_id}")
        continue
    print(f"[{index}/{len(message_ids)}] start message_id={message_id} subject={message.subject!r}", flush=True)
    started = time.perf_counter()
    try:
        result = classifier.classify(message)
        repository.save_classification(result)
        duration = time.perf_counter() - started
        timings.append(duration)
        results.append(result)
        saved_message_ids.append(message_id)
        print(f"[{index}/{len(message_ids)}] done in {duration:.2f}s category={result.category} action_type={result.action_type}", flush=True)
    except Exception as exc:
        duration = time.perf_counter() - started
        timings.append(duration)
        print(f"[{index}/{len(message_ids)}] failed in {duration:.2f}s with {type(exc).__name__}: {exc}", flush=True)

total_elapsed = time.perf_counter() - tool_started
print("\ntotal elapsed:", round(total_elapsed, 2), "s")
print("avg per message:", round(sum(timings) / len(timings), 2) if timings else 0, "s")
print("completed results:", len(results))
print("saved message ids:", saved_message_ids)


[2/5] start message_id=86f64b84-08de-4dbf-af8a-cab85929cc60 subject='Save at least 15% on your summer stay ☀️'


In [ ]:
summary_tool = EmailSummaryTool(repository=repository)
summary_result = summary_tool.execute(summary_tool.input_schema(max_items=3))
print(summary_result.model_dump(mode="json"))


In [ ]:
classified_bundle = next((bundle for bundle in repository.list_messages() if bundle.classification is not None), None)
if classified_bundle is None:
    raise ValueError("No classified messages found. Run the classifier cell first.")

message_id = classified_bundle.message.id
print("using classified message_id:", message_id)
print("classification:", classified_bundle.classification.model_dump(mode="json"))

draft_tool = DraftReplyTool(repository=repository)
draft_result = draft_tool.execute(draft_tool.input_schema(message_id=message_id))
print(draft_result.model_dump(mode="json"))


In [ ]:
approval = ApprovalItem(
    kind=ApprovalKind.WHATSAPP_NOTIFICATION,
    target_id=message_id,
    payload={
        "message_id": message_id,
        "destination": "whatsapp:+911234567890",
        "channel": "whatsapp",
        "title": "Demo Gmail notification",
        "body": "Notebook-triggered notification test.",
    },
    reason="Notebook demo approval",
    status=ApprovalStatus.PENDING,
)
repository.create_approval(approval)

notification_tool = NotificationTool(notifier=NotebookNotifier(), repository=repository)
notification_result = notification_tool.execute(notification_tool.input_schema(approval_id=approval.id))
print(notification_result.model_dump(mode="json"))


In [ ]:
classifier_model_name = os.getenv("EASY_AGENT_LLM_MODEL_NAME", "Qwen/Qwen3-1.7B")
classifier_device_map = os.getenv("EASY_AGENT_LLM_DEVICE_MAP", "cpu")
debug_prompt_path = "/tmp/mailmind_classifier_prompt.txt"

lm = LLMFactory.build_default_local_llm(
    model_name=classifier_model_name,
    device_map=classifier_device_map,
    max_new_tokens=1024,
    use_kv_chache=True,
)

lm.client.debug_prompt_path = debug_prompt_path

system_prompt = "You are a concise assistant."
user_prompt = "Give me exactly 3 short bullet points explaining what a large language model is."

result = lm.client.generate_result(system_prompt, user_prompt)

print("saved prompt:", debug_prompt_path)
print("content:", result.content)

In [ ]:
with open(debug_prompt_path, "r", encoding="utf-8") as f:
    saved_prompt_text = f.read()

print(saved_prompt_text[:1000])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

manual_tokenizer = AutoTokenizer.from_pretrained(classifier_model_name)
manual_model = AutoModelForCausalLM.from_pretrained(
    classifier_model_name,
    torch_dtype="auto",
    device_map=classifier_device_map,
)

manual_inputs = manual_tokenizer([saved_prompt_text], return_tensors="pt").to(manual_model.device)
manual_generated_ids = manual_model.generate(
    **manual_inputs,
    max_new_tokens=128,
)
manual_output_ids = manual_generated_ids[0][len(manual_inputs.input_ids[0]):].tolist()

print(manual_tokenizer.decode(manual_output_ids, skip_special_tokens=True))

In [ ]:
tokenizer.decode(res)

In [ ]:
import os
import traceback

from src.llm.factory import LLMFactory

classifier_model_name = os.getenv("EASY_AGENT_LLM_MODEL_NAME", "Qwen/Qwen3-1.7B")
classifier_device_map = os.getenv("EASY_AGENT_LLM_DEVICE_MAP", "mps")

print("model_name:", classifier_model_name)
print("device_map:", classifier_device_map)

try:
    llm = LLMFactory.build_default_local_llm(
        model_name=classifier_model_name,
        device_map=classifier_device_map,
        max_new_tokens=1024,
        use_kv_chache=True,
    )

    print("llm:", llm)
    print("client type:", type(llm.client).__name__)
    print("client model_name:", llm.client.model_name)

    system_prompt = "You are a concise assistant."
    user_prompt = "Give me exactly 3 short bullet points explaining what a large language model is."

    result = llm.client.generate_result(system_prompt, user_prompt)

    print("\ncontent:\n", result.content)
    print("\nthinking_content:\n", result.thinking_content)
    print("\nraw_text:\n", result.raw_text)

except Exception as exc:
    print("ERROR TYPE:", type(exc).__name__)
    print("ERROR:", exc)
    traceback.print_exc()